# Assistiv Systems — FEP Real Data Fetcher
**Frailty Geography Intelligence · Layer 2 · Kent & Medway · v3.0**

Fetches real NHS data from two sources and commits `kent-fep-data.json` to GitHub:
- **NHS Fingertips** — 10 outcomes indicators via `fingertips_py`
- **NHSBSA English Prescribing Dataset** — 7 prescribing signal groups, now at **practice level**

### What's new in v3.0 — Sub-ICB EPD Linkage
Previously, EPD signals were computed at ICB level then distributed to districts using
demographic multiplier profiles (educated estimates). v3.0 eliminates those multipliers
for all 7 EPD signals by computing prescribing rates **per district** from actual
practice-level data already present in the EPD.

**How:** Each EPD row contains `PRACTICE_CODE` (col 8) and `POSTCODE` (col 13).
The practice postcode outward code is mapped to a Kent district via a static lookup.
Items accumulate per district; rates = district items ÷ district list size.

### Before running:
1. Add your GitHub token to Colab Secrets (🔑 left sidebar)
   - Name: `GITHUB_TOKEN` · Value: your token with `public_repo` scope
2. `Runtime → Run all` — takes approximately 5–8 minutes

### To refresh quarterly:
- Update `EPD_URL` and `EPD_PERIOD` in Cell 1
- Latest EPD: https://opendata.nhsbsa.net/dataset/english-prescribing-dataset-epd-with-snomed-code

---
*NHSBSA Copyright 2026 · OpenPrescribing.net, Bennett Institute, University of Oxford · OHID*

## Cell 1 — Configuration
Installs dependencies and sets all constants.
Update `EPD_URL` and `EPD_PERIOD` each quarter.

**v3.0 additions:**
- `DISTRICT_LIST_SIZES` — registered patient list size per district (for rate normalisation)
- `POSTCODE_TO_DISTRICT` — maps postcode outward codes to Kent districts
- `PRACTICE_I = 8`, `POSTCODE_I = 13` — new column indices extracted from the EPD stream

In [8]:
import subprocess
subprocess.run(["pip", "install", "fingertips_py", "requests", "pandas", "--quiet"], check=True)
print("Dependencies installed")

import requests, pandas as pd, fingertips_py as ftp
import json, csv, base64
from datetime import datetime, timezone
from collections import defaultdict
from google.colab import userdata

GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-fep-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()
print(f"GitHub token: {'Found' if GITHUB_TOKEN else 'MISSING - add to Colab Secrets'}")

KENT_ICB_ONS   = "E54000032"
KENT_ICB_CODE  = "QKS"
KENT_COUNTY    = "E10000016"
ENGLAND        = "E92000001"

# ── UPDATE THESE EACH QUARTER ─────────────────────────────────────────────────
# Latest EPD: https://opendata.nhsbsa.net/dataset/english-prescribing-dataset-epd-with-snomed-code
EPD_URL    = "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/6b52baa3-c550-4b1d-8cd2-46d40d840353/download/epd_snomed_202603.csv"
EPD_PERIOD = "Mar 2026"

# England reference rates per 1,000 patients/month (OpenPrescribing 2024/25)
ENGLAND_PRESCRIBING_RATES = {
    'hypnotics':        10.2,
    'anxiolytics':       8.5,   # benzodiazepines — national published rate
    'antidepressants':  110.0,
    'bisphosphonates':   6.8,
    'diuretics':         2.5,
    'ace_arb':          95.0,
    'bladder_antimusc':  4.2,   # oxybutynin/solifenacin/tolterodine — national rate
}

# ── v3.0: EPD column indices ───────────────────────────────────────────────────
# NHSBSA EPD_SNOMED column layout (0-indexed):
#  0 YEAR_MONTH  3 ICB_NAME     6 PCN_CODE     8 PRACTICE_CODE
#  9 ADDRESS_1  13 POSTCODE    14 BNF_CHAPTER_PLUS_CODE
# 15 CHEMICAL_SUBSTANCE_BNF_DESCR   20 ITEMS   21 QUANTITY
ICB_I      = 3    # ICB_NAME (filter: 'KENT AND MEDWAY')
BNF_I      = 14   # BNF_CHAPTER_PLUS_CODE
CHEM_I     = 15   # CHEMICAL_SUBSTANCE_BNF_DESCR
PRACTICE_I = 8    # PRACTICE_CODE  ← NEW v3
POSTCODE_I = 13   # POSTCODE       ← NEW v3
ITEMS_I    = 20
QTY_I      = 21

KENT_FILTER = 'KENT AND MEDWAY'

# ── v3.0: District list sizes (NHS Digital Practice List Sizes 2024, aggregated) ──
DISTRICT_LIST_SIZES = {
    "Thanet":              145_000,
    "Folkestone & Hythe":  116_000,
    "Dover":               115_000,
    "Swale":               153_000,
    "Medway":              295_000,
    "Gravesham":           106_000,
    "Ashford":             128_000,
    "Canterbury":          172_000,
    "Dartford":            111_000,
    "Maidstone":           183_000,
    "Tonbridge & Malling": 132_000,
    "Sevenoaks":           118_000,
    "Tunbridge Wells":     119_000,
}
KENT_LIST_SIZE = sum(DISTRICT_LIST_SIZES.values())  # ICB total (kept for reference)
ALL_DISTRICTS  = list(DISTRICT_LIST_SIZES.keys())

# ── v3.0: Postcode outward code → District lookup ──────────────────────────────
# Derived from ONS NSPL + LAD 2023 boundaries. Covers all 66 outward codes
# within NHS Kent & Medway ICB. Used to route each EPD practice row to its district.
POSTCODE_TO_DISTRICT = {
    # Thanet
    'CT9':  'Thanet',   'CT10': 'Thanet',   'CT11': 'Thanet',   'CT12': 'Thanet',
    # Canterbury
    'CT1':  'Canterbury', 'CT2': 'Canterbury', 'CT3': 'Canterbury',
    'CT4':  'Canterbury', 'CT5': 'Canterbury', 'CT6': 'Canterbury',
    # Dover
    'CT13': 'Dover',  'CT14': 'Dover',  'CT15': 'Dover',  'CT16': 'Dover',  'CT17': 'Dover',
    # Folkestone & Hythe
    'CT18': 'Folkestone & Hythe', 'CT19': 'Folkestone & Hythe',
    'CT20': 'Folkestone & Hythe', 'CT21': 'Folkestone & Hythe', 'TN29': 'Folkestone & Hythe',
    # Medway (unitary authority)
    'ME1':  'Medway', 'ME2':  'Medway', 'ME3': 'Medway', 'ME4': 'Medway',
    'ME5':  'Medway', 'ME7':  'Medway', 'ME8': 'Medway',
    # Swale
    'ME9':  'Swale', 'ME10': 'Swale', 'ME11': 'Swale', 'ME12': 'Swale', 'ME13': 'Swale',
    # Maidstone
    'ME14': 'Maidstone', 'ME15': 'Maidstone', 'ME16': 'Maidstone',
    'ME17': 'Maidstone', 'ME18': 'Maidstone',
    # Tonbridge & Malling
    'ME19': 'Tonbridge & Malling', 'ME20': 'Tonbridge & Malling',
    'TN10': 'Tonbridge & Malling', 'TN11': 'Tonbridge & Malling', 'TN12': 'Tonbridge & Malling',
    # Gravesham
    'DA11': 'Gravesham', 'DA12': 'Gravesham', 'DA13': 'Gravesham',
    # Dartford
    'DA1':  'Dartford', 'DA2': 'Dartford',
    # Sevenoaks
    'TN13': 'Sevenoaks', 'TN14': 'Sevenoaks', 'TN15': 'Sevenoaks',
    'TN16': 'Sevenoaks', 'TN8':  'Sevenoaks', 'DA3':  'Sevenoaks', 'DA4': 'Sevenoaks',
    # Tunbridge Wells
    'TN1':  'Tunbridge Wells', 'TN2': 'Tunbridge Wells', 'TN3': 'Tunbridge Wells',
    'TN4':  'Tunbridge Wells', 'TN5': 'Tunbridge Wells', 'TN6': 'Tunbridge Wells',
    # Ashford
    'TN23': 'Ashford', 'TN24': 'Ashford', 'TN25': 'Ashford',
    'TN26': 'Ashford', 'TN27': 'Ashford', 'TN30': 'Ashford',
}

print(f"Config loaded | ICB: {KENT_ICB_ONS} | EPD: {EPD_PERIOD}")
print(f"District list sizes: {len(DISTRICT_LIST_SIZES)} districts, {KENT_LIST_SIZE:,} total patients")
print(f"Postcode lookup: {len(POSTCODE_TO_DISTRICT)} outward codes → {len(set(POSTCODE_TO_DISTRICT.values()))} districts")
print(f"Tracking {len(ENGLAND_PRESCRIBING_RATES)} EPD signal groups (7 BNF targets)")

Dependencies installed
GitHub token: Found
Config loaded | ICB: E54000032 | EPD: Mar 2026
Tracking 7 EPD signal groups (7 BNF targets)


## Cell 2 — Fetch NHS Fingertips Indicators
10 real outcomes indicators for Kent & Medway ICB via `fingertips_py`.
Unchanged from v1.

In [9]:
FINGERTIPS_INDICATORS = {
    'falls_65':           (22401, 'Falls admissions 65+',        KENT_ICB_ONS),
    'falls_65_79':        (22402, 'Falls admissions 65-79',      KENT_ICB_ONS),
    'falls_80':           (22403, 'Falls admissions 80+',        KENT_ICB_ONS),
    'winter_mortality':   (90360, 'Winter mortality index',      KENT_COUNTY),
    'loneliness':         (94175, 'Loneliness often/always',     KENT_ICB_ONS),
    'social_isolation':   (90280, 'Social isolation - SC users', KENT_COUNTY),
    'dementia_diagnosis': (92949, 'Dementia diagnosis rate 65+', KENT_ICB_ONS),
    'hip_fractures_65':   (41401, 'Hip fractures 65+',           KENT_ICB_ONS),
    'hip_fractures_80':   (41403, 'Hip fractures 80+',           KENT_ICB_ONS),
    'fuel_poverty':       (93759, 'Fuel poverty',                KENT_ICB_ONS),
}

fingertips_results = {}
print("Fetching NHS Fingertips indicators...")
print(f"{'Indicator':<35} {'Kent':>8}  {'England':>8}  Period")
print("-" * 70)

for key, (ind_id, label, area_code) in FINGERTIPS_INDICATORS.items():
    try:
        data = ftp.get_data_for_indicator_at_all_available_geographies(ind_id)
        if data is None:
            raise ValueError("Returned None")
        kent = data[data['Area Code'] == area_code].sort_values('Time period')
        eng  = data[data['Area Code'] == ENGLAND].sort_values('Time period')
        if len(kent) == 0:
            raise ValueError(f"No data for {area_code}")
        kent_val = round(float(kent.tail(1)['Value'].values[0]), 2)
        eng_val  = round(float(eng.tail(1)['Value'].values[0]), 2) if len(eng) else None
        period   = str(kent.tail(1)['Time period'].values[0])
        fingertips_results[key] = {
            'value': kent_val, 'england': eng_val,
            'period': period, 'source': f'NHS Fingertips indicator {ind_id}', 'label': label,
        }
        direction = "up" if eng_val and kent_val > eng_val else "down"
        print(f"  {label:<33} {kent_val:>8}  {str(eng_val):>8}  {period}  {direction}")
    except Exception as e:
        print(f"  FAILED {label:<29} -- {e}")
        fingertips_results[key] = {
            'value': None, 'england': None, 'period': None,
            'source': f'NHS Fingertips indicator {ind_id}', 'label': label,
        }

real_ft = sum(1 for v in fingertips_results.values() if v['value'])
print(f"\n{real_ft}/{len(FINGERTIPS_INDICATORS)} Fingertips indicators fetched")


Fetching NHS Fingertips indicators...
Indicator                               Kent   England  Period
----------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/fingertips_py/retrieve_data.py:164: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_temp])


  Falls admissions 65+               1917.36   1870.47  2024/25  up


/usr/local/lib/python3.12/dist-packages/fingertips_py/retrieve_data.py:164: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_temp])


  Falls admissions 65-79              959.92    863.11  2024/25  up


...

  Social isolation - SC users           49.1     46.87  2024/25  up
  Dementia diagnosis rate 65+           62.3      66.3  2026  down


/usr/local/lib/python3.12/dist-packages/fingertips_py/retrieve_data.py:164: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_temp])


  Hip fractures 65+                   542.97    491.15  2024/25  up


/usr/local/lib/python3.12/dist-packages/fingertips_py/retrieve_data.py:164: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_temp])


  Hip fractures 80+                  1414.58   1325.17  2024/25  up
  Fuel poverty                         10.97     34.59  2023  down

10/10 Fingertips indicators fetched


## Cell 3 — Stream NHSBSA EPD at Practice Level  ← v3.0 core change
Streams the EPD CSV line by line (~18M rows).
Captures 7 BNF signal groups including anxiolytics and bladder antimuscarinics.

**v3.0 change:** now accumulates items **per district** using practice postcode (col 13),
rather than at ICB level. A `practice_district_cache` avoids re-looking up the same
practice on every row (each practice appears hundreds of times per month).
ICB-level totals are also kept alongside for comparison.

**Takes 3–5 minutes** — progress shown every 2M rows.

In [10]:
# BNF prefixes to capture — 7 signal groups (unchanged from v2.1)
TARGET_BNF = (
    '0401010',   # Hypnotics (Z-drugs, zopiclone etc)
    '0401020',   # Anxiolytics (benzodiazepines — diazepam, lorazepam etc)
    '0403',      # Antidepressants (incl tricyclics with high anticholinergic burden)
    '060602',    # Bisphosphonates (bone fragility proxy)
    '0201',      # Diuretics
    '0205',      # ACE inhibitors / ARBs
    '0704010',   # Bladder antimuscarinics (oxybutynin, solifenacin, tolterodine)
)

def bnf_to_signal(bnf):
    if   bnf.startswith('0401010'): return 'hypnotics'
    elif bnf.startswith('0401020'): return 'anxiolytics'
    elif bnf.startswith('0403'):    return 'antidepressants'
    elif bnf.startswith('060602'):  return 'bisphosphonates'
    elif bnf.startswith('0201'):    return 'diuretics'
    elif bnf.startswith('0205'):    return 'ace_arb'
    elif bnf.startswith('0704010'): return 'bladder_antimusc'
    return None

# ── v3.0: District-level accumulators (replaces single ICB dict) ───────────────
district_signal_items = defaultdict(lambda: defaultdict(int))
district_signal_qty   = defaultdict(lambda: defaultdict(float))

# ICB-level accumulators kept for comparison / fallback
signal_items    = defaultdict(int)
signal_quantity = defaultdict(float)
signal_drugs    = defaultdict(set)

# Practice → district cache: populated on first encounter of each practice code
# (each practice appears hundreds of times per month — cache avoids repeated lookups)
practice_district_cache = {}
unmapped_practices = set()   # QA: practices whose postcode didn't resolve to a district
unmapped_items = 0

rows_read = kent_rows = mapped_rows = 0
buffer = ""; header_done = False

print(f"Streaming NHSBSA EPD — {EPD_PERIOD} (v3.0 practice-level mode)")
print(f"Targeting {len(TARGET_BNF)} BNF groups including anticholinergic burden signals")
print("Progress every 2M rows. Takes 3–5 minutes.\n")

try:
    with requests.get(EPD_URL, stream=True, timeout=300,
                      headers={"User-Agent": "Mozilla/5.0 AssistivSystems/3.0"}) as resp:
        print(f"HTTP {resp.status_code}")
        if resp.status_code != 200:
            raise ValueError(f"HTTP {resp.status_code} — check EPD_URL in Cell 1")
        for raw_chunk in resp.iter_content(chunk_size=512*1024):
            buffer += raw_chunk.decode('utf-8', errors='replace')
            lines = buffer.split('\n'); buffer = lines[-1]
            for line in lines[:-1]:
                line = line.strip()
                if not line: continue
                if not header_done:
                    header_done = True; continue
                rows_read += 1
                if rows_read % 2_000_000 == 0:
                    print(f"  {rows_read:>12,} rows | {kent_rows:>6,} Kent | "
                          f"{mapped_rows:>6,} mapped | {len(practice_district_cache)} practices seen")
                if KENT_FILTER not in line: continue
                try:
                    row = next(csv.reader([line]))
                except Exception:
                    continue
                if len(row) <= max(ICB_I, BNF_I, ITEMS_I, QTY_I, PRACTICE_I, POSTCODE_I): continue
                if KENT_FILTER not in row[ICB_I]: continue
                bnf = row[BNF_I].strip('"')
                if not bnf.startswith(TARGET_BNF): continue
                signal = bnf_to_signal(bnf)
                if not signal: continue
                try:
                    items         = int(row[ITEMS_I])
                    qty           = float(row[QTY_I])
                    chem          = row[CHEM_I]
                    practice_code = row[PRACTICE_I].strip()
                    postcode      = row[POSTCODE_I].strip().upper()
                except (ValueError, IndexError):
                    continue
                kent_rows += 1

                # ICB-level accumulation (v2 approach — kept for comparison)
                signal_items[signal]    += items
                signal_quantity[signal] += qty
                signal_drugs[signal].add(chem)

                # v3.0: District-level accumulation via practice postcode
                if practice_code not in practice_district_cache:
                    outward  = postcode.split()[0] if postcode else ''
                    district = POSTCODE_TO_DISTRICT.get(outward)
                    practice_district_cache[practice_code] = district
                    if district is None:
                        unmapped_practices.add(f"{practice_code}({outward})")

                district = practice_district_cache[practice_code]
                if district:
                    district_signal_items[district][signal] += items
                    district_signal_qty[district][signal]   += qty
                    mapped_rows += 1
                else:
                    unmapped_items += items

except Exception as e:
    import traceback; print(f"\nError at row {rows_read:,}: {e}"); traceback.print_exc()

print(f"\nCOMPLETE — {rows_read:,} rows scanned, {kent_rows:,} Kent matched")
print(f"  District-mapped: {mapped_rows:,} ({100*mapped_rows/max(kent_rows,1):.1f}% of Kent rows)")
print(f"  Unmapped items:  {unmapped_items:,} — expect <2% for in-ICB practices")
print(f"  Practices seen:  {len(practice_district_cache)} | Unmapped: {len(unmapped_practices)}")
if unmapped_practices:
    print(f"  Unmapped list: {sorted(unmapped_practices)[:20]}")
print(f"\nKent raw totals ({EPD_PERIOD}) — ICB level:")
for key in ['hypnotics','anxiolytics','antidepressants','bisphosphonates','diuretics','ace_arb','bladder_antimusc']:
    print(f"  {key:<22} {signal_items.get(key,0):>10,} items  ({len(signal_drugs.get(key,set()))} substances)")

Streaming NHSBSA EPD -- Mar 2026
Targeting 7 BNF groups including anticholinergic burden signals
Progress every 2M rows. Takes 3-5 minutes.

HTTP 200
     2,000,000 rows |      0 Kent | {}
     4,000,000 rows | 12,878 Kent | {'anxiolytics': 2338, 'antidepressants': 49963, 'bisphosphonates': 3109, 'bladder_antimusc': 5685, 'diuretics': 1187, 'ace_arb': 44106, 'hypnotics': 4466}
     6,000,000 rows | 16,692 Kent | {'anxiolytics': 3128, 'antidepressants': 60819, 'bisphosphonates': 3813, 'bladder_antimusc': 7195, 'diuretics': 1455, 'ace_arb': 55258, 'hypnotics': 5670}
     8,000,000 rows | 25,747 Kent | {'anxiolytics': 4765, 'antidepressants': 93571, 'bisphosphonates': 6125, 'bladder_antimusc': 11671, 'diuretics': 2188, 'ace_arb': 86407, 'hypnotics': 9298}
    10,000,000 rows | 29,041 Kent | {'anxiolytics': 5275, 'antidepressants': 107276, 'bisphosphonates': 6911, 'bladder_antimusc': 13274, 'diuretics': 2517, 'ace_arb': 97770, 'hypnotics': 11417}
    12,000,000 rows | 36,852 Kent | {'anxio

## Cell 4 — Calculate District-Level Prescribing Rates  ← v3.0 core change
Converts raw item counts to rates per 1,000 patients using **district-specific list sizes**.
Also computes ICB-level rates (v2 approach) alongside for comparison.

**QA check:** District rates should form a plausible geographic gradient.
Thanet and Folkestone & Hythe typically sit above ICB average; Sevenoaks and
Tunbridge Wells typically below. Ratios > 2.0 or < 0.3 vs England warrant investigation.

In [11]:
EPD_SIGNAL_KEYS = [
    'hypnotics', 'anxiolytics', 'antidepressants',
    'bisphosphonates', 'diuretics', 'ace_arb', 'bladder_antimusc'
]

# ── ICB-level rates (v2 approach — kept for comparison and icb_baseline output) ──
epd_results = {}
print(f"ICB-level rates (v2 approach — for comparison):")
print(f"  {'Signal':<22} {'Kent/1k':>9}  {'Eng/1k':>9}  {'Ratio':>7}")
print(f"  {'-'*55}")
for key in EPD_SIGNAL_KEYS:
    items  = signal_items.get(key, 0)
    qty    = signal_quantity.get(key, 0.0)
    drugs  = len(signal_drugs.get(key, set()))
    kent_r = round((items / KENT_LIST_SIZE) * 1000, 3) if items else 0
    eng_r  = ENGLAND_PRESCRIBING_RATES.get(key, 0)
    ratio  = round(kent_r / eng_r, 3) if eng_r and kent_r else 0
    estimate_flag = " [est]" if key in ('anxiolytics', 'bladder_antimusc') else ""
    epd_results[key] = {
        'rate_per_1000': kent_r, 'england': eng_r, 'items': items,
        'qty': round(qty), 'substances': drugs, 'ratio': ratio,
        'period': EPD_PERIOD,
        'source': f'NHSBSA EPD EPD_SNOMED_{EPD_PERIOD.replace(" ","").upper()}',
        'england_note': 'literature estimate' if estimate_flag else 'OpenPrescribing 2024/25',
    }
    print(f"  {key:<22} {kent_r:>9.3f}  {eng_r:>9.1f}{estimate_flag}  {ratio:>7.3f}")

# ── v3.0: District-level rates ─────────────────────────────────────────────────
print(f"\nDistrict prescribing rates per 1,000 patients (v3.0 — actual local data):")
header = f"  {'District':<25} {'List':>7}"
for s in EPD_SIGNAL_KEYS:
    header += f"  {s[:7]:>8}"
print(header)
print(f"  {'-'*110}")

epd_district_results = {}   # epd_district_results[district][signal]

for district in ALL_DISTRICTS:
    list_size = DISTRICT_LIST_SIZES[district]
    epd_district_results[district] = {}
    row_str = f"  {district:<25} {list_size:>7,}"
    for key in EPD_SIGNAL_KEYS:
        items  = district_signal_items[district].get(key, 0)
        rate   = round((items / list_size) * 1000, 3) if items else 0
        eng_r  = ENGLAND_PRESCRIBING_RATES.get(key, 0)
        ratio  = round(rate / eng_r, 3) if eng_r and rate else 0
        epd_district_results[district][key] = {
            'rate_per_1000': rate, 'england': eng_r,
            'items': items, 'ratio': ratio,
            'period': EPD_PERIOD,
            'source': f'NHSBSA EPD practice-level — {EPD_PERIOD}',
        }
        flag = "▲" if ratio > 1.05 else "▼" if ratio < 0.95 else "="
        row_str += f"  {rate:>7.2f}{flag}"
    print(row_str)

# QA: flag implausible district rates
print("\nQA — district signals with ratio > 2.0 or < 0.3 vs England:")
issues = []
for district in ALL_DISTRICTS:
    for key in EPD_SIGNAL_KEYS:
        r = epd_district_results[district][key].get('ratio', 0)
        if r > 2.0 or (0 < r < 0.3):
            issues.append(f"  *** {district} / {key}: ratio {r:.3f} — review list size or BNF filter")
if issues:
    for i in issues: print(i)
else:
    print("  All district rates within plausible range ✓")

real_epd = sum(1 for v in epd_results.values() if v['rate_per_1000'] > 0)
total_real = sum(1 for v in fingertips_results.values() if v['value']) + real_epd
print(f"\n{real_epd}/{len(EPD_SIGNAL_KEYS)} EPD signals computed | Total real signals: {total_real}/17")

# Sanity check on anxiolytics and bladder_antimusc (literature-derived England rates)
for key in ('anxiolytics', 'bladder_antimusc'):
    r = epd_results.get(key, {}).get('ratio', 0)
    if r > 2.0 or (r > 0 and r < 0.3):
        print(f"  *** REVIEW: {key} ICB ratio {r} looks unusual — check England reference rate in Cell 1 ***")
    elif r > 0:
        print(f"  {key} ICB ratio {r:.3f} — plausible range confirmed")

Kent & Medway prescribing rates vs England (Mar 2026)
List size: 1,900,000

  Signal                   Kent/1k     Eng/1k    Ratio  Direction
  --------------------------------------------------------------
  hypnotics                 12.189       10.2    1.195  above Eng
  anxiolytics                6.192        8.5 [est]    0.728  below Eng
  antidepressants          129.445      110.0    1.177  above Eng
  bisphosphonates            8.288        6.8    1.219  above Eng
  diuretics                  2.912        2.5    1.165  above Eng
  ace_arb                  118.616       95.0    1.249  above Eng
  bladder_antimusc          15.679        4.2 [est]    3.733  above Eng

7/7 EPD signals computed
Total real signals: 17/17
  anxiolytics ratio 0.728 — plausible range confirmed

  *** REVIEW: bladder_antimusc ratio 3.733 looks unusual — check England reference rate in Cell 1 ***


## Cell 5 — Build District FEP Scores & Commit to GitHub
Normalises signals to 0–100, computes weighted FEP scores, commits `kent-fep-data.json`.

**v3.0 changes:**
- EPD signals (indices 10–16) now use `epd_district_results[district][signal]` — actual
  local prescribing rates — instead of `ICB_BASE × PROFILE_multiplier`
- `PROFILES` multiplier is now only applied to non-EPD signals (Fingertips + synthetics)
- EPD columns in PROFILES are set to `1.0` (ignored) to make this explicit
- Scorecard at end compares v3.0 vs previous committed scores

In [13]:
# ── SIGNAL DEFINITIONS (17 signals — unchanged from v2.1) ───────────────────
SIGNAL_NAMES = [
    "Over-75s Living Alone",       # 0   synthetic
    "Falls Admissions 65+",        # 1   Fingertips real
    "Hip Fracture Rate 65+",       # 2   Fingertips real
    "Deprivation (IMD)",           # 3   synthetic
    "Winter Mortality Index",      # 4   Fingertips real
    "Care Home Gap",               # 5   synthetic
    "Loneliness Rate",             # 6   Fingertips real
    "Dementia Diagnosis Rate",     # 7   Fingertips real (inverted)
    "Hip Fractures 80+",           # 8   Fingertips real
    "Social Isolation Rate",       # 9   Fingertips real
    "Hypnotics Prescribing",       # 10  EPD ← DISTRICT-LEVEL in v3
    "Antidepressant Rate",         # 11  EPD ← DISTRICT-LEVEL in v3
    "Bisphosphonate Rate",         # 12  EPD ← DISTRICT-LEVEL in v3
    "Diuretics Rate",              # 13  EPD ← DISTRICT-LEVEL in v3
    "ACE/ARB Prescribing",         # 14  EPD ← DISTRICT-LEVEL in v3
    "Anxiolytics Prescribing",     # 15  EPD ← DISTRICT-LEVEL in v3
    "Bladder Antimuscarinic Rate", # 16  EPD ← DISTRICT-LEVEL in v3
]

# Signal indices that use district EPD data (vs ICB Fingertips + profiles)
EPD_SIGNAL_INDICES      = list(range(10, 17))
EPD_SIGNAL_KEYS_ORDERED = [
    'hypnotics', 'antidepressants', 'bisphosphonates',
    'diuretics', 'ace_arb', 'anxiolytics', 'bladder_antimusc'
]

WEIGHTS = [
    0.13,  # alone_75
    0.12,  # falls_65
    0.09,  # hip_65
    0.08,  # deprivation
    0.08,  # winter_mortality
    0.07,  # care_home_gap
    0.06,  # loneliness
    0.06,  # dementia
    0.05,  # hip_80
    0.05,  # social_isolation
    0.05,  # hypnotics
    0.05,  # antidepressants
    0.04,  # bisphosphonates
    0.01,  # diuretics
    0.01,  # ace_arb
    0.03,  # anxiolytics
    0.02,  # bladder_antimusc
]
assert abs(sum(WEIGHTS) - 1.0) < 0.001, f"Weights sum to {sum(WEIGHTS):.4f} — must be 1.0"
print(f"Weight check: {sum(WEIGHTS):.4f} ✓")

def norm(value, england, invert=False):
    if not value or not england: return 50.0
    score = (value / england) * 50
    return round(min(100, max(0, 100 - score if invert else score)), 1)

def ft(key, invert=False):
    v = fingertips_results.get(key, {})
    return norm(v.get('value'), v.get('england'), invert)

def epd_for_district(district, signal_key):
    """v3.0: normalised score from actual district-level prescribing rate."""
    v = epd_district_results.get(district, {}).get(signal_key, {})
    return norm(v.get('rate_per_1000'), ENGLAND_PRESCRIBING_RATES.get(signal_key))

# ICB-level non-EPD base scores (Fingertips + synthetics)
# EPD slots (indices 10-16) are 0.0 here — overwritten per district below
ICB_BASE_NONEEPD = [
    50.0,                                    # alone_75 — synthetic
    ft('falls_65'),                          # falls_65
    ft('hip_fractures_65'),                  # hip_65
    50.0,                                    # deprivation — synthetic
    ft('winter_mortality'),                  # winter_mortality
    50.0,                                    # care_home_gap — synthetic
    ft('loneliness'),                        # loneliness
    ft('dementia_diagnosis', invert=True),   # dementia (lower = worse)
    ft('hip_fractures_80'),                  # hip_80
    ft('social_isolation'),                  # social_isolation
    0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,     # 10-16: EPD — filled per district
]

print("\nICB normalised scores for non-EPD signals (England avg = 50):")
real_count = 0
for i, (name, score) in enumerate(zip(SIGNAL_NAMES[:10], ICB_BASE_NONEEPD[:10])):
    is_real = score != 50.0
    if is_real: real_count += 1
    tag = "REAL  " if is_real else "synth "
    bar = ">" if score > 50 else "<" if score < 50 else "="
    print(f"  {bar} {score:>5.1f}  [{tag}]  {name}")
print(f"\n  Non-EPD real signals: {real_count}/10")

# ── PROFILES: multiplier now only applied to non-EPD signals (cols 0–9) ───────
# Columns 10–16 are set to 1.0 — EPD uses real district data, not multipliers
PROFILES = {
    "Thanet":              [1.30,1.25,1.20,1.35,1.28,1.30,1.25,1.18,1.20,1.22, 1,1,1,1,1,1,1],
    "Folkestone & Hythe":  [1.22,1.18,1.15,1.22,1.20,1.20,1.18,1.12,1.15,1.15, 1,1,1,1,1,1,1],
    "Dover":               [1.18,1.15,1.12,1.18,1.15,1.10,1.14,1.08,1.10,1.10, 1,1,1,1,1,1,1],
    "Swale":               [1.12,1.10,1.08,1.12,1.10,1.12,1.08,1.05,1.08,1.05, 1,1,1,1,1,1,1],
    "Medway":              [1.06,1.05,1.04,1.08,1.05,1.08,1.02,1.02,1.05,1.02, 1,1,1,1,1,1,1],
    "Gravesham":           [1.00,0.98,1.02,1.02,1.00,1.05,0.98,1.00,1.02,1.00, 1,1,1,1,1,1,1],
    "Ashford":             [0.96,0.95,0.98,0.98,0.96,1.00,0.94,0.96,0.98,0.95, 1,1,1,1,1,1,1],
    "Canterbury":          [0.90,0.90,0.92,0.85,0.92,0.92,0.92,0.90,0.90,0.90, 1,1,1,1,1,1,1],
    "Dartford":            [0.88,0.88,0.90,0.95,0.88,0.90,0.88,0.88,0.88,0.88, 1,1,1,1,1,1,1],
    "Maidstone":           [0.85,0.85,0.88,0.95,0.85,0.92,0.85,0.85,0.85,0.85, 1,1,1,1,1,1,1],
    "Tonbridge & Malling": [0.78,0.78,0.82,0.78,0.80,0.85,0.80,0.80,0.80,0.80, 1,1,1,1,1,1,1],
    "Sevenoaks":           [0.65,0.65,0.68,0.52,0.65,0.75,0.65,0.65,0.65,0.65, 1,1,1,1,1,1,1],
    "Tunbridge Wells":     [0.58,0.58,0.62,0.58,0.60,0.65,0.60,0.60,0.60,0.60, 1,1,1,1,1,1,1],
}

LAD_CODES = {
    "Thanet":"E07000114", "Folkestone & Hythe":"E07000112", "Dover":"E07000108",
    "Swale":"E07000113",  "Medway":"E06000035",  "Gravesham":"E07000109",
    "Ashford":"E07000105","Canterbury":"E07000106","Dartford":"E07000107",
    "Maidstone":"E07000110","Tonbridge & Malling":"E07000115",
    "Sevenoaks":"E07000111","Tunbridge Wells":"E07000116",
}

POP75 = {
    "Thanet":18200,"Folkestone & Hythe":14100,"Dover":13800,"Swale":15200,
    "Medway":19400,"Gravesham":11800,"Ashford":13600,"Canterbury":16300,
    "Dartford":10800,"Maidstone":16700,"Tonbridge & Malling":13100,
    "Sevenoaks":12100,"Tunbridge Wells":11200,
}

# ── BUILD DISTRICT SCORES ──────────────────────────────────────────────────────
districts = []
print("\nDistrict FEP scores (v3.0 — district-level EPD):")
print(f"  {'District':<25} {'FEP':>5}  {'Risk':<10}  antidep  hypnot  anxiol")
print("  " + "-" * 70)

for name, profile in PROFILES.items():
    # Non-EPD signals: ICB base × demographic profile (unchanged from v2)
    signals = [
        round(min(100, max(0, ICB_BASE_NONEEPD[i] * profile[i])), 1)
        for i in range(17)
    ]
    # EPD signals: replace with actual district-level normalised rates
    for idx, epd_key in zip(EPD_SIGNAL_INDICES, EPD_SIGNAL_KEYS_ORDERED):
        signals[idx] = epd_for_district(name, epd_key)

    fep  = round(min(100, max(0, sum(s * w for s, w in zip(signals, WEIGHTS)))))
    risk = "critical" if fep >= 70 else "high" if fep >= 55 else "moderate" if fep >= 40 else "low"
    marker = " ◀ CRITICAL" if risk == 'critical' else ""
    # Show 3 key EPD signals for spot-check
    ad = epd_district_results[name]['antidepressants']['rate_per_1000']
    hy = epd_district_results[name]['hypnotics']['rate_per_1000']
    ax = epd_district_results[name]['anxiolytics']['rate_per_1000']
    print(f"  {name:<25} {fep:>5}  {risk:<10}  {ad:>7.2f}  {hy:>6.2f}  {ax:>6.2f}{marker}")

    districts.append({
        "name":         name,
        "lad_code":     LAD_CODES[name],
        "fep":          fep,
        "risk":         risk,
        "signals":      signals,
        "signal_names": SIGNAL_NAMES,
        "pop75":        POP75[name],
        "list_size":    DISTRICT_LIST_SIZES[name],
        "epd_district": {k: epd_district_results[name][k] for k in EPD_SIGNAL_KEYS},
    })

districts.sort(key=lambda x: x["fep"], reverse=True)

# ── Scorecard: v3.0 vs v2.1 ───────────────────────────────────────────────────
print("\n── v3.0 vs previous scores (fetching from GitHub) ──")
try:
    prev = requests.get(
        f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/{GITHUB_FILE}",
        timeout=10
    ).json()
    prev_scores = {d['name']: d['fep'] for d in prev.get('districts', [])}
    prev_sorted = sorted(prev.get('districts', []), key=lambda x: x['fep'], reverse=True)
    v2_ranks = {d['name']: i+1 for i, d in enumerate(prev_sorted)}
    v3_ranks = {d['name']: i+1 for i, d in enumerate(districts)}
    print(f"  {'District':<25} {'prev':>5}  {'v3.0':>5}  {'Δ':>5}  Move")
    print(f"  {'-'*52}")
    for d in districts:
        n  = d['name']
        v2 = prev_scores.get(n, '?')
        v3 = d['fep']
        delta     = (v3 - v2) if isinstance(v2, int) else '?'
        rank_move = v2_ranks.get(n, 0) - v3_ranks.get(n, 0)
        arrow = f"↑{rank_move}" if rank_move > 0 else (f"↓{abs(rank_move)}" if rank_move < 0 else "─")
        print(f"  {n:<25} {str(v2):>5}  {v3:>5}  {str(delta):>5}  {arrow}")
except Exception as e:
    print(f"  (Could not fetch previous scores: {e})")

# ── ASSEMBLE & COMMIT ──────────────────────────────────────────────────────────
real_signals = (
    [k for k, v in fingertips_results.items() if v.get('value')] +
    [k for k, v in epd_results.items() if v.get('rate_per_1000', 0) > 0]
)

output = {
    "meta": {
        "generated":         datetime.now(timezone.utc).isoformat(),
        "description":       "Kent & Medway FEP scores — Assistiv Systems Layer 2",
        "version":           "3.0 — sub-ICB EPD linkage at practice level",
        "icb":               "NHS Kent and Medway ICB (QKS)",
        "icb_ons_code":      KENT_ICB_ONS,
        "data_quality":      f"real — {len(real_signals)} real signals of 17 total",
        "signals_real":      real_signals,
        "signals_synthetic": ["alone_75", "deprivation_imd", "care_home_gap"],
        "signal_names":      SIGNAL_NAMES,
        "weights":           WEIGHTS,
        "sources": {
            "fingertips":    "NHS Fingertips/OHID PHOF via fingertips_py",
            "epd":           f"NHSBSA EPD practice-level — {EPD_PERIOD}",
            "epd_method":    "Practice POSTCODE (col 13) → district via POSTCODE_TO_DISTRICT",
            "demographic":   "ONS Census 2021 · IMD 2019 · CQC register",
        },
        "new_in_v3": [
            "EPD signals computed at district level using practice POSTCODE (col 13)",
            f"practice_district_cache: {len(practice_district_cache)} practices mapped",
            f"unmapped practices: {len(unmapped_practices)} (expect <5 for in-ICB practices)",
            "PROFILES multiplier retained only for non-EPD signals (indices 0–9)",
            "DISTRICT_LIST_SIZES replace single KENT_LIST_SIZE for EPD rate normalisation",
        ],
        "note": "EPD: district-level actual. Fingertips + synthetic: ICB × demographic profile. Phase 2: MSOA Fingertips linkage.",
    },
    "icb_baseline": {
        "fingertips":  fingertips_results,
        "prescribing": epd_results,
    },
    "districts": districts,
}

def commit_to_github(content, repo, filepath, token):
    api_url = f"https://api.github.com/repos/{repo}/contents/{filepath}"
    headers = {"Authorization": "token " + token,
               "Accept": "application/vnd.github.v3+json"}
    b64 = base64.b64encode(json.dumps(content, indent=2).encode()).decode()
    r = requests.get(api_url, headers=headers)
    sha = r.json().get("sha") if r.status_code == 200 else None
    payload = {
        "message": f"FEP v3.0 — {datetime.now(timezone.utc).strftime('%Y-%m-%d')} — practice-level EPD ({len(practice_district_cache)} practices)",
        "content": b64,
        "branch":  "main",
    }
    if sha: payload["sha"] = sha
    r = requests.put(api_url, headers=headers, json=payload)
    if r.status_code in (200, 201):
        print(f"\nCommitted to GitHub ✓")
        print(f"  File:   https://raw.githubusercontent.com/{GITHUB_REPO}/main/{GITHUB_FILE}")
        print(f"  Commit: {r.json().get('commit',{}).get('html_url','')}")
        return True
    print(f"\nCommit failed: {r.status_code} — {r.json().get('message','')}")
    return False

print("\nCommitting to GitHub...")
commit_to_github(output, GITHUB_REPO, GITHUB_FILE, GITHUB_TOKEN)
print("\nDone. Run quarterly to refresh.")
print(f"Practices mapped: {len(practice_district_cache)} | Unmapped: {len(unmapped_practices)}")

Weight check: 1.0000 ✓

ICB normalised scores (England avg = 50):
  =  50.0  [synth ]  Over-75s Living Alone
  >  51.3  [REAL  ]  Falls Admissions 65+
  >  55.3  [REAL  ]  Hip Fracture Rate 65+
  =  50.0  [synth ]  Deprivation (IMD)
  >  56.3  [REAL  ]  Winter Mortality Index
  =  50.0  [synth ]  Care Home Gap
  <  26.6  [REAL  ]  Loneliness Rate
  >  53.0  [REAL  ]  Dementia Diagnosis Rate
  >  53.4  [REAL  ]  Hip Fractures 80+
  >  52.4  [REAL  ]  Social Isolation Rate
  >  59.8  [REAL  ]  Hypnotics Prescribing
  >  58.8  [REAL  ]  Antidepressant Rate
  >  60.9  [REAL  ]  Bisphosphonate Rate
  >  58.2  [REAL  ]  Diuretics Rate
  >  62.4  [REAL  ]  ACE/ARB Prescribing
  <  36.4  [REAL  ]  Anxiolytics Prescribing
  > 100.0  [REAL  ]  Bladder Antimuscarinic Rate

  Real signals: 14/17

FINAL DISTRICT FEP SCORES (v2 — 14 real signals, 17 total):
  District                    FEP  Risk
  ----------------------------------------
  Thanet                       65  high
  Folkestone & Hythe 